# 14a · Creare il servizio Azure Update Service per il namespace ADR

Gli **aggiornamenti software** di ADR non li fa il namespace: li fa un servizio a parte, **Azure Update Service** (nome della risorsa: *Update Instance*, tipo `Microsoft.DeviceUpdate/updateInstances`, in preview).
Il namespace ADR si **collega** a una Update Instance. Da quel momento:
- **tu** importi un aggiornamento (file + import manifest) nella Update Instance;
- il **job** del namespace dice a quale gruppo darlo;
- il **device** chiede l'aggiornamento sulla sua connessione DPS (con il suo certificato); ADR inoltra la richiesta alla Update Instance. Il device non parla mai con il servizio direttamente.

```text
  tu ---- az iot adr ns su software-update import ----> Update Instance <---- link ---- Namespace ADR ---- job/run ----> gruppo
                                                                                              ^
                                       device ---- DPS (X.509): "c'e' un aggiornamento per me?" ---+
```

Questo notebook crea quello che serve **prima** del primo aggiornamento:
1. la **Update Instance** (con identità di sistema);
2. il **collegamento** al namespace ADR (creazione asincrona: la Update Instance ha impiegato **più di 15 minuti** a lasciare *Creating*, il 2026-11-22);
3. il tuo ruolo **Device Update Content Administrator** (per importare);
4. uno **storage account** con un container, dove stanno i file dell'aggiornamento, e i ruoli per leggerli.

**Serve:** i notebook 01 e 02a-02c già eseguiti, e l'estensione `azure-iot` con i comandi `az iot adr ns su ...` (vedi la cella dei controlli).
**Costo:** storage account e Update Instance restano nel resource group e si eliminano con il notebook 99.

**Da dire.** "Il namespace non sa aggiornare: ha bisogno di un servizio collegato. È il pezzo che di solito si dimentica di mettere nello schema."


In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

## Controlli (sola lettura)
Servono il provider `Microsoft.DeviceUpdate` registrato e i comandi `su` nell'estensione. Non crea niente.


In [ ]:
az provider show --namespace Microsoft.DeviceUpdate --query registrationState -o tsv --only-show-errors
az iot adr ns su instance -h --only-show-errors | Select-String 'create|check-name' | Select-Object -First 3


## 1 · I nomi e l'identità del namespace
I nomi vengono da `_scenario.psd1` (`$update`): Update Instance `adu-<workload>-<stage>-<regione>-<istanza>`, storage `st<workload><stage><regione><istanza>`.
Il namespace deve avere un'**identità di sistema**: il collegamento la usa per parlare con la Update Instance senza credenziali.


In [ ]:
$update | Format-Table | Out-String
$ident = az iot adr ns show --name $adrName --resource-group $rg --query identity.type -o tsv --only-show-errors
"Identita' del namespace: $ident"
if ($ident -notmatch 'SystemAssigned') {
    az iot adr ns identity assign --name $adrName --resource-group $rg --system-assigned true -o none --only-show-errors
    Assert-Az "Assegnazione dell'identita' di sistema al namespace"
}


## 2 · La Update Instance
Una risorsa **a livello di resource group**, non figlia del namespace. La creazione è asincrona e può durare diversi minuti: la cella aspetta.


In [ ]:
$esiste = az iot adr ns su instance show --name $update.Name --resource-group $rg --query properties.provisioningState -o tsv --only-show-errors 2>$null
if ($esiste -eq 'Succeeded') { "Update Instance $($update.Name) gia' presente." }
else {
    az iot adr ns su instance create `
        --name $update.Name `
        --resource-group $rg `
        --location $rgRegion `
        --system-assigned-mi true `
        -o none `
        --only-show-errors
    Assert-Az "Creazione della Update Instance $($update.Name)"
}
$suId = az iot adr ns su instance show --name $update.Name --resource-group $rg --query id -o tsv --only-show-errors
Assert-Az "Lettura della Update Instance"
"Update Instance: $suId"


## 3 · Il collegamento al namespace ADR
**Un solo** collegamento per namespace. Quando riesce, il collegamento riporta l'indirizzo del servizio (`serviceAddress`), quello che usano gli import.


In [ ]:
$link = az iot adr ns link su show --name $update.Link --ns $adrName --resource-group $rg --only-show-errors 2>$null
if (-not $link) {
    az iot adr ns link su add `
        --name $update.Link `
        --ns $adrName `
        --resource-group $rg `
        --su-id $suId `
        --system-assigned-mi true `
        --timeout 1800 `
        -o none `
        --only-show-errors
    Assert-Az "Collegamento della Update Instance al namespace"
}
az iot adr ns link su list --ns $adrName --resource-group $rg -o json --only-show-errors


## 4 · Il tuo ruolo per importare
Importare un aggiornamento richiede **Device Update Content Administrator** sulla Update Instance. Il ruolo di Owner della subscription non basta: è un ruolo del piano dati.


In [ ]:
$io = az ad signed-in-user show --query id -o tsv --only-show-errors
Assert-Az "Lettura dell'utente connesso"
az role assignment create `
    --assignee-object-id $io `
    --assignee-principal-type User `
    --role 'Device Update Content Administrator' `
    --scope $suId `
    -o none `
    --only-show-errors
Assert-Az "Assegnazione del ruolo Device Update Content Administrator"
"Ruolo assegnato a te sulla Update Instance (puo' servire qualche minuto per essere attivo)."


## 5 · Lo storage per i file dell'aggiornamento
Un account con un container privato. L'import legge i file da qui. **Storage Blob Data Contributor** serve a te (per caricare) e **Storage Blob Data Reader** alla Update Instance (per leggere).
Il comando `stage` usa la chiave dell'account: per questo la chiave condivisa resta abilitata (demo, non produzione).


In [ ]:
az storage account create `
    --name $update.Storage `
    --resource-group $rg `
    --location $rgRegion `
    --sku Standard_LRS `
    --allow-blob-public-access false `
    -o none `
    --only-show-errors
Assert-Az "Creazione dello storage account $($update.Storage)"
$stId = az storage account show --name $update.Storage --resource-group $rg --query id -o tsv --only-show-errors
az role assignment create --assignee-object-id $io --assignee-principal-type User --role 'Storage Blob Data Contributor' --scope $stId -o none --only-show-errors
Assert-Az "Ruolo Storage Blob Data Contributor per te"
$suPrincipal = az iot adr ns su instance show --name $update.Name --resource-group $rg --query identity.principalId -o tsv --only-show-errors
az role assignment create --assignee-object-id $suPrincipal --assignee-principal-type ServicePrincipal --role 'Storage Blob Data Reader' --scope $stId -o none --only-show-errors
Assert-Az "Ruolo Storage Blob Data Reader per la Update Instance"
az storage container create --name $update.Container --account-name $update.Storage --auth-mode login -o none --only-show-errors
if ($LASTEXITCODE) { "Il ruolo non e' ancora attivo: riesegui questa riga tra un minuto (il container e' l'ultimo passo)." }
"Storage $($update.Storage), container $($update.Container)."


## 6 · Il collegamento è attivo?
Finché il collegamento non ha l'indirizzo del servizio (`serviceAddress`) gli import non funzionano. La cella aspetta fino a 30 minuti.


In [ ]:
az iot adr ns link su wait --name $update.Link --ns $adrName --resource-group $rg --timeout 1800 --only-show-errors
Assert-Az "Attesa del collegamento"
az iot adr ns link su show --name $update.Link --ns $adrName --resource-group $rg -o json --only-show-errors


## Cosa abbiamo visto
- Gli aggiornamenti software sono un **servizio a parte**: Azure Update Service, una **Update Instance** collegata al namespace ADR (una sola per namespace).
- Il collegamento **non è immediato** (fino a 30 minuti) e usa l'identità di sistema, senza credenziali salvate.
- Chi importa serve un **ruolo del piano dati** (Device Update Content Administrator), e i file stanno in uno **storage tuo**.
- Il device **non parla con questo servizio**: chiede su DPS, ADR inoltra.

**Prossimo notebook: 14, lo stato della preview e i comandi veri; poi 15 (gruppi) e 16.**
